# Milestone 7 — SHAP explanations

Explain the selected base model on a reproducible sample of validation records. XGBoost SHAP contributions are in raw log-odds, not percentage points or causal effects. The grouped Type contribution sums its one-hot contributions. Use tree-path-dependent Tree SHAP, whose reference distribution follows training path counts. Persist the trusted explainer for identical serving behavior. This avoids the installed SHAP/XGBoost interventional-mode compatibility issue without changing the trained model.

In [1]:
from pathlib import Path
import nbformat
PROJECT_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "data/raw/ai4i2020.csv").exists())
notebook = nbformat.read(PROJECT_ROOT / "notebooks/04_baseline_models.ipynb", as_version=4)
for cell in notebook.cells:
    if cell.cell_type == "code" and "definitions" in cell.metadata.get("tags", []):
        exec(compile(cell.source, "baseline_definitions", "exec"), globals())
from xgboost import XGBClassifier
records, manifest = load_partitions()
X_train, y_train = get_partition(records, manifest, "train")
X_validation, y_validation = get_partition(records, manifest, "validation")

In [2]:
import shap
base_pipeline = joblib.load(MODEL_DIR / "explanation_model.joblib")
metadata = json.loads((MODEL_DIR / "metadata.json").read_text())
preprocessor = base_pipeline.named_steps["preprocess"]
classifier = base_pipeline.named_steps["classifier"]
explainer = shap.TreeExplainer(classifier, model_output="raw", feature_perturbation="tree_path_dependent")
sample = X_validation.sample(n=400, random_state=RANDOM_STATE)
transformed = preprocessor.transform(sample)
explanations = explainer(transformed)
values = np.asarray(explanations.values)
assert values.shape == transformed.shape
raw_margin = classifier.predict(transformed, output_margin=True)
np.testing.assert_allclose(np.asarray(explanations.base_values) + values.sum(axis=1), raw_margin, atol=1e-4)
grouped = np.column_stack([values[:, :5], values[:, 5:].sum(axis=1)])
feature_names = [*NUMERIC_COLUMNS, "Type"]
importance = [{"feature": name, "mean_absolute_shap": float(value)} for name, value in zip(feature_names, np.abs(grouped).mean(axis=0))]
importance.sort(key=lambda item: item["mean_absolute_shap"], reverse=True)
report = {"model_version": metadata["model_version"], "units": "log_odds", "sample": "400 validation rows, seed 42", "feature_importance": importance, "interpretation": "Contributed to the model score; not a causal explanation"}
(METRICS_DIR / "feature_importance.json").write_text(json.dumps(report, indent=2) + "\n")
joblib.dump(explainer, MODEL_DIR / "explainer.joblib")
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), layout="constrained")
axes[0].barh([item["feature"] for item in importance][::-1], [item["mean_absolute_shap"] for item in importance][::-1], color="#287c9b")
axes[0].set(xlabel="Mean absolute contribution (log-odds)", title="Global model explanation")
individual = grouped[0]
axes[1].barh(feature_names, individual, color=["#b64732" if v > 0 else "#287c9b" for v in individual])
axes[1].axvline(0,color="gray")
axes[1].set(xlabel="Contribution to raw model score (log-odds)", title="One validation observation")
fig.savefig(FIGURE_DIR / "shap_explanations.png",dpi=160)
display(fig)
plt.close(fig)
display(pd.DataFrame(importance))
example = {"inputs": sample.iloc[0].to_dict(), "failure_probability": float(base_pipeline.predict_proba(sample.iloc[[0]])[0,1]), "base_value": float(explanations.base_values[0]), "units": "log_odds", "contributions": dict(zip(feature_names, individual.tolist()))}
(METRICS_DIR / "explanation_example.json").write_text(json.dumps(example,indent=2) + "\n")
metadata["explanation_units"] = "log_odds"
metadata["artifact_hashes"] = {name: hashlib.sha256((MODEL_DIR / name).read_bytes()).hexdigest() for name in ("production.joblib", "explanation_model.joblib", "explainer.joblib")}
(MODEL_DIR / "metadata.json").write_text(json.dumps(metadata,indent=2) + "\n")
print("SHAP additivity verified against the model raw margin.")

/run/media/jadeite/Local Disk/Predictive Maintenance AI Platform/.venv/lib64/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


<Figure size 1300x450 with 2 Axes>

,feature,mean_absolute_shap
0,Torque [Nm],1.692756
1,Tool wear [min],1.348852
2,Air temperature [K],1.053296
3,Rotational speed [rpm],1.008648
4,Process temperature [K],0.424002
5,Type,0.194799


SHAP additivity verified against the model raw margin.
